In [12]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from loss import CombinedLoss
from model import DeeperSegmentationCNN
from dataset import CerebellumSliceDataset

In [13]:
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib

In [14]:
from collections import defaultdict

In [15]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
test_dataset = CerebellumSliceDataset(dataset_split="test")
test_loader = DataLoader(dataset=test_dataset, batch_size=4, shuffle=False, num_workers=0)

deeper_cnn = DeeperSegmentationCNN().to(device)
deeper_cnn.load_state_dict(torch.load("best_deeper_cerebellum_model.pth", map_location=device))

volume_predictions = {}

total_tp = 0.0
total_fp = 0.0
total_fn = 0.0

deeper_cnn.eval()

with torch.no_grad():
    for images, masks, patients, slice_indices in test_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        predictions = deeper_cnn(images)
        binarized_predictions = (predictions > 0.5).float()
        binarized_masks = (masks > 0.5).float()
        
        flat_predictions = binarized_predictions.view(-1)
        flat_masks = binarized_masks.view(-1)
        
        # Accumulate global pixel metrics
        total_tp += (flat_predictions * flat_masks).sum().item()
        total_fp += (flat_predictions * (1.0 - flat_masks)).sum().item()
        total_fn += ((1.0 - flat_predictions) * flat_masks).sum().item()
        
        # Convert predictions to NumPy
        preds_np = binarized_predictions.cpu().squeeze(1).numpy()
        
        for i, patient_id in enumerate(patients):
            pid = patient_id if isinstance(patient_id, str) else patient_id[i]
            if pid not in volume_predictions:
                volume_predictions[pid] = []
            volume_predictions[pid].append(preds_np[i])

smooth = 1e-6
test_dice = (2.0 * total_tp + smooth) / (2.0 * total_tp + total_fp + total_fn + smooth)
test_iou = (total_tp + smooth) / (total_tp + total_fp + total_fn + smooth)

print(f"\n--- Final Test Evaluation Results ---")
print(f"Test Set Dice Score: {test_dice:.4f}")
print(f"Test Set IoU:       {test_iou:.4f}")


--- Final Test Evaluation Results ---
Test Set Dice Score: 0.8418
Test Set IoU:       0.7269


In [16]:
precision = (total_tp + smooth) / (total_tp + total_fp + smooth)
recall = (total_tp + smooth) / (total_tp + total_fn + smooth)

In [17]:
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")

Test Precision: 0.8755
Test Recall: 0.8106


In [18]:
# output_dir = "predicted_nifti_outputs"
# os.makedirs(output_dir, exist_ok=True)

# print(f"Exporting predicted 3D volumes to '{output_dir}'...")

In [19]:
# for pid, slice_list in volume_predictions.items():
#     volume_3d = np.stack(slice_list, axis=-1) # Stack the list of 2D arrays (shape: H, W) along the z-axis to form a 3D volume (H, W, Z)
    
#     volume_3d = volume_3d.astype(np.float32)
  
#     affine = np.eye(4) 
  
#     pred_nii = nib.Nifti1Image(volume_3d, affine)
  
#     output_filename = os.path.join(output_dir, f"{pid}_cerebellum_pred.nii.gz")
#     nib.save(pred_nii, output_filename)
#     print(f"Saved: {output_filename} with shape {volume_3d.shape}")

# print("Export complete! You can now load these .nii.gz files directly into 3D Slicer or ITK-SNAP.")

In [20]:
# for pid, slice_list in volume_predictions.items():
#     num_slices = len(slice_list)
    
#     # Select up to 16 evenly spaced slices across the patient's volume
#     step = max(1, num_slices // 16)
#     selected_indices = list(range(0, num_slices, step))[:16]
    
#     # Set up a grid layout (4 columns)
#     cols = 4
#     rows = (len(selected_indices) + cols - 1) // cols
#     fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    
#     # Flatten axes array to handle indexing easily
#     axes_flat = np.atleast_1d(axes).flatten()
    
#     for idx, slice_idx in enumerate(selected_indices):
#         ax = axes_flat[idx]
#         ax.imshow(slice_list[slice_idx], cmap='gray')
#         ax.set_title(f"Slice {slice_idx}")
#         ax.axis('off')
        
#     # Hide any unused slots in the subplot grid
#     for idx in range(len(selected_indices), len(axes_flat)):
#         axes_flat[idx].axis('off')
        
#     plt.suptitle(f"Patient ID: {pid} ({num_slices} Total Slices)", fontsize=14, y=1.02)
#     plt.tight_layout()
#     plt.show()

In [21]:
import cv2

In [22]:
target_filenames = [
    ("patient4", "OASIS-TRT-20-4_coronal-059.png"),
    ("patient4", "OASIS-TRT-20-4_coronal-105.png"),
    ("patient1", "OASIS-TRT-20-1_coronal-039.png"), 
    ("patient1", "OASIS-TRT-20-1_coronal-094.png")
]

deeper_cnn.eval()

with torch.no_grad():
    for folder, file in target_filenames:
        img_path = os.path.join("../data_processed", "images", "test", folder, file)
        mask_path = os.path.join("../data_processed", "masks", "test", folder, file)
        
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        raw_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        img_norm = raw_img.astype(np.float32) / 255.0
        mask_norm = raw_mask.astype(np.float32) / 255.0
        
        img_tensor = torch.from_numpy(img_norm).unsqueeze(0).unsqueeze(0).to(device)
        
        mask_tensor = torch.from_numpy(mask_norm).to(device)
        mask_tensor = (mask_tensor > 0.5).float().view(-1)
        
        pred_logits = deeper_cnn(img_tensor)
        pred_binary = (pred_logits > 0.5).float().view(-1)
        
        tp = (pred_binary * mask_tensor).sum().item()
        fp = (pred_binary * (1.0 - mask_tensor)).sum().item()
        fn = ((1.0 - pred_binary) * mask_tensor).sum().item()
        
        dice_score = (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)
        print(f"{folder} | {file} | Dice: {dice_score:.4f}")
        
        pred_2d = pred_binary.view(raw_img.shape).cpu().numpy()
        
        pred_img_to_save = (pred_2d * 255).astype(np.uint8)
        
        save_dir = os.path.join(".", "test_cases", "test", folder)
        os.makedirs(save_dir, exist_ok=True)
        
        save_path = os.path.join(save_dir, f"pred_{file}")
        cv2.imwrite(save_path, pred_img_to_save)

patient4 | OASIS-TRT-20-4_coronal-059.png | Dice: 0.9124
patient4 | OASIS-TRT-20-4_coronal-105.png | Dice: 0.6283
patient1 | OASIS-TRT-20-1_coronal-039.png | Dice: 0.0000
patient1 | OASIS-TRT-20-1_coronal-094.png | Dice: 0.7712
